# Basic_박상우_7주차_1

4.3, 4.4 정리

## 4.3 Voting Classifier

In [ ]:
import pandas as pd

from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 유방암 데이터셋을 불러와서 전체 구조만 먼저 확인
cancer = load_breast_cancer()
data_df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
data_df.head(3)

In [ ]:
# 개별 모델은 로지스틱 회귀와 KNN 두 개만 사용
lr_clf = LogisticRegression(solver='liblinear')
knn_clf = KNeighborsClassifier(n_neighbors=8)

# soft voting이라서 각 모델의 확률값을 평균내서 최종 예측
vo_clf = VotingClassifier(
    estimators=[('LR', lr_clf), ('KNN', knn_clf)],
    voting='soft'
)

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=156
)

vo_clf.fit(X_train, y_train)
pred = vo_clf.predict(X_test)
print('Voting 분류기 정확도: {0:.4f}'.format(accuracy_score(y_test, pred)))

# 앙상블이랑 개별 모델 성능을 같이 비교
classifiers = [lr_clf, knn_clf]
for classifier in classifiers:
    classifier.fit(X_train, y_train)
    pred = classifier.predict(X_test)
    class_name = classifier.__class__.__name__
    print('{0} 정확도: {1:.4f}'.format(class_name, accuracy_score(y_test, pred)))

## 4.4 Random Forest

In [ ]:
def get_new_feature_name_df(old_feature_name_df):
    # 같은 이름이 여러 번 나오면 뒤에 번호를 붙여서 겹치지 않게 만든다.
    feature_dup_df = pd.DataFrame(
        data=old_feature_name_df.groupby('column_name').cumcount(),
        columns=['dup_cnt']
    )
    feature_dup_df = feature_dup_df.reset_index()
    new_feature_name_df = pd.merge(old_feature_name_df.reset_index(), feature_dup_df, how='outer')
    new_feature_name_df['column_name'] = new_feature_name_df[['column_name', 'dup_cnt']].apply(
        lambda x: x[0] + '_' + str(x[1]) if x[1] > 0 else x[0], axis=1
    )
    new_feature_name_df = new_feature_name_df.drop(['index'], axis=1)
    return new_feature_name_df

In [ ]:
import warnings
warnings.filterwarnings('ignore')

def get_human_dataset():
    # features.txt에서 컬럼 이름을 먼저 읽는다.
    feature_name_df = pd.read_csv(
        './human_activity/features.txt',
        sep='\s+',
        header=None,
        names=['column_index', 'column_name']
    )

    new_feature_name_df = get_new_feature_name_df(feature_name_df)
    feature_name = new_feature_name_df.iloc[:, 1].values.tolist()

    # 학습 데이터와 테스트 데이터를 같은 컬럼 구조로 읽어온다.
    X_train = pd.read_csv('./human_activity/train/X_train.txt', sep='\s+', names=feature_name)
    X_test = pd.read_csv('./human_activity/test/X_test.txt', sep='\s+', names=feature_name)
    y_train = pd.read_csv('./human_activity/train/y_train.txt', sep='\s+', header=None, names=['action'])
    y_test = pd.read_csv('./human_activity/test/y_test.txt', sep='\s+', header=None, names=['action'])
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = get_human_dataset()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 기본 랜덤 포레스트로 먼저 성능 확인
rf_clf = RandomForestClassifier(random_state=0)
rf_clf.fit(X_train, y_train.values.ravel())
pred = rf_clf.predict(X_test)
accuracy = accuracy_score(y_test.values.ravel(), pred)

print('Random Forest 정확도: {0:.4f}'.format(accuracy))

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    'n_estimators': [100],
    'max_depth': [6, 8, 10, 12],
    'min_samples_leaf': [8, 12, 18],
    'min_samples_split': [8, 16, 20]
}

# 여러 조합을 돌려보고 가장 잘 나오는 파라미터를 찾는다.
rf_clf = RandomForestClassifier(random_state=0, n_jobs=-1)
grid_cv = GridSearchCV(rf_clf, param_grid=params, cv=2, n_jobs=-1)
grid_cv.fit(X_train, y_train.values.ravel())

print('최적 하이퍼파라미터:', grid_cv.best_params_)
print('최고 예측 정확도: {0:.4f}'.format(grid_cv.best_score_))

In [ ]:
rf_clf1 = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=8,
    min_samples_split=8,
    random_state=0
)

# GridSearchCV 결과를 반영해서 다시 학습
rf_clf1.fit(X_train, y_train.values.ravel())
pred = rf_clf1.predict(X_test)
print('튜닝 후 정확도: {0:.4f}'.format(accuracy_score(y_test.values.ravel(), pred)))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# 중요도가 높은 상위 20개 피처만 따로 본다.
ftr_importances_values = rf_clf1.feature_importances_
ftr_importances = pd.Series(ftr_importances_values, index=X_train.columns)
ftr_top20 = ftr_importances.sort_values(ascending=False)[:20]

plt.figure(figsize=(8, 6))
plt.title('Feature importances Top 20')
sns.barplot(x=ftr_top20, y=ftr_top20.index)
plt.show()